In [ ]:
# this notebook is for assembling model inputs for conversation modeling (GPT embeddings as well as spike counts/frs)

In [1]:
# module imports
from glob import glob
import pandas as pd
import os
from scipy.io import loadmat
import scipy.sparse as sp
import h5py

In [ ]:
# set dirs
data_dir = "/mnt/labworlds/Hayden/Hayden_Lab/speech_247/podcast_comparison"

In [3]:
# get patient dirs
patient_dirs = glob(f"{data_dir}/Y*")
patient_dirs = {os.path.basename(dir_):dir_ for dir_ in patient_dirs}

In [4]:
# load spikes and transcripts
patient_spikes = {}
for pt, dir_ in patient_dirs.items():
    spike_path = glob(f"{dir_}/spikes/*.mat")[0]
    with h5py.File(spike_path) as f:
        sparse_mat = f["spikes"]
        data = sparse_mat["data"][:].flatten()  # Extract nonzero values
        ir = sparse_mat["ir"][:].flatten()      # Row indices
        jc = sparse_mat["jc"][:].flatten()      # Co
    
        # Determine the shape of the matrix (from jc array)
        num_rows = int(ir.max() + 1)  # Max row index + 1
        num_cols = len(jc) - 1   # The number of columns (jc's length - 1)
    
        # Create the scipy.sparse.csc_matrix
        sparse_matrix = sp.csc_matrix((data, ir, jc), shape=(num_rows, num_cols))
    spiketrain = sparse_matrix.toarray()
    patient_spikes[pt] = spiketrain

In [5]:
# load transcripts
patient_transcripts = {}
for pt, dir_ in patient_dirs.items():
    transcript_path = glob(f"{dir_}/transcript/*")[0]
    transcript = pd.read_excel(transcript_path)
    patient_transcripts[pt] = transcript
    

In [6]:
import numpy as np
from scipy.signal import fftconvolve

def gaussian_firing_rate(
    spike_bin,          # (n_channels, n_bins) uint8/float; 1s where spikes occur
    fs=1000,            # Hz; your spike_bin rate
    sigma_ms=50.0,      # Gaussian std in milliseconds (typical: 10–100 ms)
    truncate=3.0,       # kernel half-width in sigmas (3→ ~99.7% mass)
    edge_mode="reflect" # 'reflect', 'constant', 'nearest'; for padding
):
    """
    Return: rates_hz with shape (n_channels, n_bins), smoothed firing rate in Hz.
    """
    spike_bin = np.asarray(spike_bin, dtype=float)
    n_ch, n_bins = spike_bin.shape

    # build Gaussian kernel in bins
    sigma_bins = sigma_ms * fs / 1000.0
    radius = int(np.ceil(truncate * sigma_bins))
    t = np.arange(-radius, radius + 1, dtype=float)
    kernel = np.exp(-0.5 * (t / sigma_bins) ** 2)
    kernel /= kernel.sum()  # area = 1 (in bins)

    # pad for edge handling
    def pad1d(x):
        if edge_mode == "reflect":
            left = x[:, 1:radius+1][:, ::-1]
            right = x[:, -radius-1:-1][:, ::-1]
        elif edge_mode == "nearest":
            left = np.repeat(x[:, :1], radius, axis=1)
            right = np.repeat(x[:, -1:], radius, axis=1)
        elif edge_mode == "constant":
            left = np.zeros((x.shape[0], radius))
            right = np.zeros((x.shape[0], radius))
        else:
            raise ValueError("edge_mode must be 'reflect', 'nearest', or 'constant'")
        return np.concatenate([left, x, right], axis=1)

    xpad = pad1d(spike_bin)

    # FFT convolution per channel
    rates = np.empty_like(spike_bin, dtype=float)
    for c in range(n_ch):
        y = fftconvolve(xpad[c], kernel, mode="same")
        # unpad back to original length
        rates[c] = y[:,][radius:-radius]

    # convert from spikes/bin to spikes/s
    rates_hz = rates * fs
    return rates_hz

In [7]:
# now get spike counts/frs from word onset to offset
patient_frs = {}
patient_counts = {}
for pt, df in patient_transcripts.items():
    word_frs = []
    word_counts = []
    spikes = patient_spikes[pt]
    gaussian_frs = gaussian_firing_rate(spikes)
    
    for idx, row in df.iterrows():
        # now get onset/offset index of word
        word_start = np.round(row.onset).astype(int)
        word_end = np.round(row.offset).astype(int)

        # now get spike counts and frs
        word_spikes = spikes[:, word_start:word_end].sum(axis=1)
        frs = gaussian_frs[:, word_start:word_end].mean(axis=1)

        word_counts.append(word_spikes)
        word_frs.append(frs)

    word_frs = np.vstack(word_frs)
    word_counts = np.vstack(word_counts)
    patient_frs[pt] = word_frs
    patient_counts[pt] = word_counts

In [ ]:
# now get GPT-2 embeddings for each word
# reset context of df
from collections import deque
context_size = 200
context_window = deque()
for pt, df in patient_transcripts.items():
    context_window = deque()
    for idx, row in df.iterrows():
        # get current context
        context = " ".join(context_window)
        df.loc[idx, 'context'] = context
        # update context window
        # if idx != 0 and filter_df.loc[idx - 1]['recording_id'] != row.recording_id:
        #     context_window = deque()
        if isinstance(row.text, str) and (len(context_window) == 0 or row.text != context_window[-1]):
            context_window.append(row.text)
        if len(context_window) > context_size:
            context_window.popleft()
    df['context'] = df['context'].astype(str)
    df['context'] = df['context'].apply(lambda x: x if x != 'nan' else '')
    patient_transcripts[pt] = df

In [9]:
# login to hugginface
from huggingface_hub import login

# Replace "hf_YOUR_ACCESS_TOKEN" with your actual token
login(token=os.environ['HF_TOKEN'])

In [10]:
# device
import torch
device = torch.device("cuda")

In [11]:
# set huggingface home and load gemma model
from transformers import GPT2Tokenizer, GPT2Model
import torch

os.environ["HF_HOME"] = "/scratch/ti12/hf"
tokenizer = GPT2Tokenizer.from_pretrained('gpt2-large')
model = GPT2Model.from_pretrained('gpt2-large').to(device).eval()


In [12]:
def rfind_sublist(lst, sub):
    """Return the starting index of the last occurrence of `sub` in `lst`, or -1 if not found."""
    n, m = len(lst), len(sub)
    for i in range(n - m, -1, -1):  # start from last possible match
        if lst[i:i + m] == sub:
            return i
    return -1

In [13]:
# potentially add option to remove punctuation while searching
def get_average_embedding(embeddings, token_list, word):
    # starting from end of token list, do linear search for the word we're looking for
    # tokenize our particular word
    enc = tokenizer(word, return_tensors='pt', add_special_tokens=True)
    tok_rep = tokenizer.convert_ids_to_tokens(enc.input_ids[0])
    # remove [CLS] [SEP] tokens
    tok_rep = tok_rep
    idx = rfind_sublist(token_list, tok_rep)
    word_embedding = np.nanmean(embeddings[:, idx:idx+len(tok_rep), :], axis=1)
    return word_embedding
    

In [14]:
from tqdm import tqdm
patient_embeddings = {}
# now get embeddings for each word
model.eval()
for pt, df in patient_transcripts.items():
    embeddings = []
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        try:
            text = row.context + ' ' + row.text if (isinstance(row.context, str) and row.context != '') else row.text
            enc = tokenizer(text, return_tensors='pt', add_special_tokens=True).to(device)
            # try to output embedding, otherwise just nan vector
            with torch.no_grad():
                model_out = model(**enc, output_hidden_states=True)
                hidden_states = np.array([layer.squeeze().cpu().numpy() for layer in model_out[2]])
                if len(hidden_states.shape) == 2:
                    embedding = hidden_states
                else:
                    # get tokens
                    all_toks = tokenizer.convert_ids_to_tokens(enc.input_ids[0])
                    to_embed = row.text if row.context == '' else ' ' + row.text
                    embedding = get_average_embedding(hidden_states, all_toks, to_embed)
        except Exception as e:
            print(e)
            embedding = np.full_like(embedding, np.nan)
        embeddings.append(embedding)
    embeddings = np.array(embeddings)
    patient_embeddings[pt] = embeddings

100%|██████████| 7346/7346 [03:24<00:00, 35.84it/s]


In [15]:
# load semantic category classifier
import dill as pickle
model_dir = "/scratch/ti12/speech_247/vad_out"
with open(f"{model_dir}/semantic_cat_classifier.pkl", "rb") as f:
    model = pickle.load(f)

In [17]:
# load fasttext to get embeddings for these words
import fasttext
import fasttext.util


ft = fasttext.load_model(f'{model_dir}/cc.en.300.bin')

In [18]:
# now add semantic categories to patient transcripts
import string
for pt, df in patient_transcripts.items():
    w2v_embeddings = []
    for idx, row in df.iterrows():
        try:
            word_format = row.text.rstrip(string.punctuation).lower()
            embedding = ft.get_word_vector(word_format)
            # try to output embedding, otherwise just nan vector
        except Exception as e:
            print(e)
            embedding = np.full_like(embedding, np.nan)
        w2v_embeddings.append(embedding)
    w2v_embeddings = np.vstack(w2v_embeddings)

    threshold = 0.8
    data_probs = model.predict_proba(w2v_embeddings)
    top_idx = data_probs.argmax(axis=1)
    conf = data_probs[np.arange(data_probs.shape[0]), top_idx]
    keep = conf >= threshold
    top_idx[~keep] = -1
    df["cluster_pred"] = top_idx
    patient_transcripts[pt] = df

/projects/bhayden/ti12/conda_envs/speech_247_env/lib/python3.11/site-packages/xgboost/core.py:2676: UserWarning: [12:26:56] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  if len(data.shape) != 1 and self.num_features() != data.shape[1]:
/projects/bhayden/ti12/conda_envs/speech_247_env/lib/python3.11/site-packages/xgboost/core.py:729: UserWarning: [12:26:56] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [24]:
# now save all our stuff
for pt, spikes in patient_frs.items():
    np.save(f"{data_dir}/{pt}/spikes/word_spikes.npy", spikes)
    frs = patient_frs[pt]
    np.save(f"{data_dir}/{pt}/spikes/word_frs.npy", frs)
    embeddings = patient_embeddings[pt]
    np.save(f"{data_dir}/{pt}/embeddings/word_embeddings_gpt2.npy", embeddings)
    # also save semantic categories df
    df = patient_transcripts[pt]
    df.to_csv(f"{data_dir}/{pt}/semantic_cats/semantic_cats.csv")